# Interactive Brokers Paper Trading Demo

**Chapter**: 25 - Live Trading Systems
**Section Reference**: 25.2 (Integrating with Interactive Brokers)
**Learning Outcome**: LO2 - Connect ml4t-backtest strategies to live brokers

**Docker image**: `ml4t` (requires IB TWS/Gateway running on host port 7497)

This notebook demonstrates:
1. Connecting to IB TWS paper trading account
2. Querying account information and positions
3. Requesting historical data for strategy warm-up
4. Real-time data feed with tick aggregation
5. Safe order submission in shadow mode

**Learning Objectives**:
- Validate IB connectivity, account state, and warm-up data before a live engine starts.
- Reuse the same backtest strategy inside a shadow-mode live workflow.
- Exercise the loud-fail setup checklist: the notebook does not synthesize live behaviour from history when TWS/Gateway is unreachable or the market is closed.

**Prerequisites**:
- IB TWS or Gateway running with API enabled
- Paper trading port: 7497 (TWS) or 4002 (Gateway)
- `IB_ACCOUNT` environment variable set to your IB paper account ID (e.g. `DU1234567`); the notebook passes `None` to `IBBroker` if unset, which lets the broker pick the default account on the session.
- Familiarity with the ETF momentum example used throughout the live-trading chapter

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# Interactive Brokers 模擬交易示範

**章節**：第 25 章－實盤交易系統
**節次參考**：第 25.2 節（與 Interactive Brokers 整合）
**學習成果**：LO2－將 ml4t-backtest 策略連接至實際券商

**Docker 映像檔**：`ml4t`（主機的 7497 連接埠需有 IB TWS／Gateway 執行）

本 Notebook 示範：
1. 連接 IB TWS 模擬交易帳戶
2. 查詢帳戶資訊與部位
3. 請求歷史資料供策略暖身
4. 使用即時資料來源並彙整逐筆報價
5. 在影子模式下安全地送出委託

**學習目標**：
- 在啟動實盤引擎之前，驗證 IB 連線、帳戶狀態及暖身資料。
- 在影子模式實盤流程中重用相同的回測策略。
- 執行遇錯即明確停止的設定檢查清單：TWS／Gateway 無法連線或市場休市時，Notebook 不會用歷史資料假造即時行為。

**先備條件**：
- IB TWS 或 Gateway 已執行，且已啟用 API
- 模擬交易連接埠：7497（TWS）或 4002（Gateway）
- `IB_ACCOUNT` 環境變數已設為 IB 模擬交易帳戶 ID（例如 `DU1234567`）；若未設定，Notebook 會將 `None` 傳入 `IBBroker`，由券商選擇該工作階段的預設帳戶。
- 熟悉實盤交易章節中使用的 ETF 動能範例
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""Connect ml4t strategies to IB with shadow-mode risk controls."""

import os

`MARKET_DATA_TYPE` decides which quote TWS returns. Leaving it at `None` keeps whatever the
session is configured for, which is the right choice when the paper account carries live
Level 1 subscriptions. A paper account without them rejects MARKET orders with "No market
data available...", so this notebook asks for delayed quotes instead. The four values TWS
accepts are 1 for real time, 2 for frozen, 3 for delayed and 4 for delayed frozen.

<!-- ML4T_ZH_BEGIN c0002-p000 -->
`MARKET_DATA_TYPE` 決定 TWS 回傳哪種報價。保持 `None` 會沿用工作階段的既有設定；模擬交易帳戶若有即時 Level 1 訂閱，這就是適當選擇。若無訂閱，模擬交易帳戶會以「No market data available...」拒絕 MARKET 委託，因此本 Notebook 改為請求延遲報價。TWS 接受的四個值分別為 1（即時）、2（凍結）、3（延遲）及 4（延遲凍結）。
<!-- ML4T_ZH_END c0002-p000 -->


In [2]:
# Production defaults. Papermill may inject overrides for CI.
IB_HOST = "127.0.0.1"
IB_PORT = 7497  # Paper trading port
CLIENT_ID = 10  # Use unique ID per notebook
ACCOUNT = os.environ.get(
    "IB_ACCOUNT"
)  # set IB_ACCOUNT=DU... before running; None picks the session default
MARKET_DATA_TYPE: int | None = 3  # delayed quotes; see above
SYMBOLS = ["SPY", "QQQ", "IWM"]  # ETFs to monitor
WARMUP_DAYS = 10  # Must provide at least lookback + 1 daily closes
LIVE_DURATION_SECONDS = 75

In [3]:
import asyncio
import logging
import warnings
from datetime import datetime

os.environ.setdefault("NUMEXPR_MAX_THREADS", "16")

from async_utils import run_async

from utils.paths import display_path, get_output_dir

In [4]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

# Reduce noise from ib_async
logging.getLogger("ib_async").setLevel(logging.WARNING)
logging.getLogger("ml4t.live.brokers.ib").setLevel(logging.WARNING)

In [5]:
# Import ml4t.live components
from ml4t.backtest import OrderSide, Strategy
from ml4t.live import LiveEngine, LiveRiskConfig
from ml4t.live.brokers.ib import IBBroker
from ml4t.live.feeds import BarAggregator, IBDataFeed
from ml4t.live.safety import SafeBroker

print("[OK] ml4t.live components imported successfully")

[OK] ml4t.live components imported successfully


The import check runs before any Interactive Brokers connection is attempted, so an environment
that is missing a package fails differently from a session that cannot reach TWS. Collapsing the
two into one opaque error is what makes a live deployment slow to debug.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
匯入檢查會在嘗試連接 Interactive Brokers 前執行，因此缺少套件與無法連接 TWS 會以不同方式失敗。若將兩者合併為一個不明確的錯誤，實盤部署的除錯就會變慢。
<!-- ML4T_ZH_END c0007-p000 -->


## 1. Connect to Interactive Brokers

TWS/Gateway must be running with:
- API access enabled (Edit > Global Configuration > API > Settings)
- Socket port set (default: 7497 for paper)
- "Enable ActiveX and Socket Clients" checked

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 1. 連接 Interactive Brokers

TWS／Gateway 必須已執行，並符合以下設定：
- 已啟用 API 存取（Edit > Global Configuration > API > Settings）
- 已設定 Socket 連接埠（模擬交易預設為 7497）
- 已勾選「Enable ActiveX and Socket Clients」
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
# Create IBBroker instance
broker = IBBroker(
    host=IB_HOST,
    port=IB_PORT,
    client_id=CLIENT_ID,
    account=ACCOUNT,
    market_data_type=MARKET_DATA_TYPE,
)

print(f"IBBroker configured for {IB_HOST}:{IB_PORT}")
print(f"Account selection: {'configured paper account' if ACCOUNT else 'session default'}")
print(f"Client ID: {CLIENT_ID}")

IBBroker configured for 127.0.0.1:7497
Account selection: session default
Client ID: 10


Host, port, account and client ID are printed rather than left as constants in a cell above,
because an account or session mismatch is one of the most common live failures and is invisible
until something has already been submitted to the wrong place.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
主機、連接埠、帳戶與用戶端 ID 會直接印出，而非只作為上方儲存格的常數。帳戶或工作階段不一致是最常見的實盤故障之一，而且在委託已送至錯誤位置前，往往難以察覺。
<!-- ML4T_ZH_END c0010-p000 -->


In [7]:
async def connect_to_ib():
    """Connect to IB and show account summary."""
    print("\n" + "=" * 60)
    print("CONNECTING TO INTERACTIVE BROKERS")
    print("=" * 60)

    await broker.connect()

    account = str(broker._account or "")
    if not account.startswith("DU"):
        await broker.disconnect()
        raise RuntimeError("Refusing to continue: connected IB account is not a paper account")

    print("\n[OK] Connected to IB")
    print("   Paper account identity: [OK]")
    # Get account values
    nlv = await broker.get_account_value_async()
    cash = await broker.get_cash_async()

    print("\nACCOUNT READINESS")
    print("   Account values received: [OK]")

    # Get positions
    positions = await broker.get_positions_async()
    print(f"   Open positions received: {len(positions)}")

    return nlv, cash, positions

In [8]:
# Run connection
try:
    warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
    nlv, cash, positions = run_async(connect_to_ib())
except Exception as exc:
    print()
    print("=" * 60)
    print("ERROR: IB paper session unreachable")
    print("=" * 60)
    print(f"Could not connect to {IB_HOST}:{IB_PORT}")
    print(f"Underlying error: {type(exc).__name__}: {exc}")
    print()
    print("Setup checklist:")
    print("  1. Start TWS or IB Gateway and log into a paper account.")
    print("  2. Edit -> Global Configuration -> API -> Settings:")
    print("     - 'Enable ActiveX and Socket Clients' must be checked.")
    print(f"     - Socket port must be {IB_PORT} (TWS paper=7497, Gateway paper=4002).")
    print("     - 127.0.0.1 must be in 'Trusted IPs', or 'Read-Only API' unchecked.")
    print("  3. Confirm no other client is using this client_id.")
    print()
    print("Re-run this notebook once TWS is reachable.")
    raise RuntimeError("IB paper session unreachable") from exc


CONNECTING TO INTERACTIVE BROKERS



[OK] Connected to IB
   Paper account identity: [OK]

ACCOUNT READINESS
   Account values received: [OK]
   Open positions received: 0


When TWS is unreachable the connection block prints a checklist and stops, rather than
substituting placeholder data and letting the rest of the notebook read as if it had traded.
A disconnected broker is an operational state, which is why the check comes before the signal
loop rather than inside it.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
無法連接 TWS 時，連線區塊會印出檢查清單並停止，而不會代入佔位資料，讓 Notebook 後續看起來彷彿真的交易過。券商斷線是作業狀態，因此檢查安排在訊號循環之前，而非循環之中。
<!-- ML4T_ZH_END c0013-p000 -->


## 2. Request Historical Data

Before running a strategy live, we need historical data to:
- Calculate initial indicator values (moving averages, etc.)
- Establish baseline for position sizing
- Verify data quality

IB provides historical bars via `reqHistoricalData`.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 2. 請求歷史資料

即時執行策略之前，需要歷史資料來：
- 計算指標初始值（移動平均線等）
- 建立部位規模的基準
- 驗證資料品質

IB 透過 `reqHistoricalData` 提供歷史價格長條資料。
<!-- ML4T_ZH_END c0014-p000 -->


In [9]:
from ib_async import Stock


async def get_historical_data(symbol: str, days: int = 5) -> list[dict]:
    """Request historical daily bars from IB and return OHLCV records."""
    contract = Stock(symbol, "SMART", "USD")
    qualified = await broker.ib.qualifyContractsAsync(contract)
    if not qualified:
        raise RuntimeError(f"IB could not qualify the {symbol} contract")

    # `timeout=0` disables ib_async's internal `asyncio.wait_for` wrapper,
    # which fails under nest_asyncio on Python 3.14.
    bars = await broker.ib.reqHistoricalDataAsync(
        contract,
        endDateTime="",
        durationStr=f"{days} D",
        barSizeSetting="1 day",
        whatToShow="TRADES",
        useRTH=True,
        timeout=0,
    )

    result = []
    for bar in bars:
        result.append(
            {
                "timestamp": bar.date,
                "open": bar.open,
                "high": bar.high,
                "low": bar.low,
                "close": bar.close,
                "volume": bar.volume,
            }
        )

    if not result:
        raise RuntimeError(f"IB returned no warm-up bars for {symbol}")
    return result

In [10]:
# Request historical data for our symbols
print("\n" + "=" * 60)
print("HISTORICAL DATA (Warm-up)")
print("=" * 60)

historical_data = {}
try:
    for symbol in SYMBOLS:
        warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
        bars = run_async(get_historical_data(symbol, WARMUP_DAYS))
        if len(bars) < 6:
            raise RuntimeError(f"{symbol} returned {len(bars)} bars; at least 6 are required")
        historical_data[symbol] = bars
        latest = bars[-1]
        print(f"\n{symbol}: {len(bars)} days of data")
        print(
            f"   Latest: {latest['timestamp']} - Close: ${latest['close']:.2f}, Volume: {latest['volume']:,}"
        )
except Exception:
    warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
    run_async(broker.disconnect())
    raise

assert set(historical_data) == set(SYMBOLS)


HISTORICAL DATA (Warm-up)



SPY: 10 days of data
   Latest: 2026-07-23 - Close: $739.91, Volume: 6,032,786.0



QQQ: 10 days of data
   Latest: 2026-07-23 - Close: $694.94, Volume: 5,368,493.0



IWM: 10 days of data
   Latest: 2026-07-23 - Close: $291.95, Volume: 4,450,643.0


The warm-up prints the exact bars the indicators are initialized from. Without a deterministic
warm-up path, the first minutes after a reconnect or an open size positions from indicators that
are still filling.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
暖身階段會印出用來初始化指標的確切價格長條資料。若沒有確定性的暖身流程，重新連線或開盤後的前幾分鐘，部位規模就會根據尚未收集完整資料的指標計算。
<!-- ML4T_ZH_END c0017-p000 -->


## 3. Strategy Definition

This strategy is **identical** to what we use in backtesting.
The `on_data` signature works with both `ml4t.backtest.Engine` and `ml4t.live.LiveEngine`.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
## 3. 策略定義

這項策略與回測使用的策略**完全相同**。`on_data` 函式簽章可同時用於 `ml4t.backtest.Engine` 與 `ml4t.live.LiveEngine`。
<!-- ML4T_ZH_END c0018-p000 -->


In [11]:
class MomentumStrategy(Strategy):
    """Five-day ETF momentum strategy shared by backtest and live engines."""

    def __init__(self, lookback: int = 5, threshold: float = 0.02):
        self.lookback = lookback
        self.threshold = threshold
        self.prices: dict[str, list[float]] = {}
        self.signals: list[dict] = []

    def on_start(self, broker):
        """Called when engine starts."""
        logger.info(f"Strategy started: Momentum({self.lookback}, {self.threshold})")
        for symbol, bars in historical_data.items():
            self.prices[symbol] = [bar["close"] for bar in bars]
            logger.info(f"  {symbol}: Loaded {len(self.prices[symbol])} historical prices")

    def on_data(self, timestamp: datetime, data: dict, context: dict, broker):
        """Update trailing momentum and route threshold crossings."""
        for symbol, bar in data.items():
            prices = self.prices.setdefault(symbol, [])
            close = bar["close"]
            prices.append(close)
            if len(prices) <= self.lookback:
                continue
            momentum = (close - prices[-self.lookback - 1]) / prices[-self.lookback - 1]
            position = broker.get_position(symbol)
            has_position = position is not None and position.quantity > 0
            side = None
            if momentum > self.threshold and not has_position:
                side = OrderSide.BUY
            elif momentum < -self.threshold and has_position:
                side = OrderSide.SELL
            if side is None:
                continue
            action = side.value.upper()
            self.signals.append(
                {"timestamp": timestamp, "symbol": symbol, "action": action, "momentum": momentum}
            )
            logger.info(f"{action} {symbol}: momentum {momentum:.2%}")
            broker.submit_order(symbol, 100, side=side)

    def on_end(self, broker):
        """Called when engine stops."""
        logger.info(f"Strategy ended. Signals generated: {len(self.signals)}")

## 4. Safe Broker Configuration

Before going live, we wrap the broker with `SafeBroker` which provides:
- Shadow mode (virtual orders routed through `VirtualPortfolio`, never to IB)
- Position and order value caps, rate limiting, kill switch
- Persisted `RiskState`, so the daily-loss counter is read back after a restart
- Startup reconciliation: `safe_broker.connect()` diffs the persisted snapshot from the previous run
  against the broker's current positions and pending orders

<!-- ML4T_ZH_BEGIN c0020-p000 -->
## 4. 安全券商設定

進入實盤前，先以 `SafeBroker` 包裝券商，提供以下功能：
- 影子模式（虛擬委託透過 `VirtualPortfolio` 處理，絕不送往 IB）
- 部位與委託金額上限、速率限制、緊急停止開關
- 儲存 `RiskState`，使每日虧損計數器能在重新啟動後讀回
- 啟動時核對：`safe_broker.connect()` 比較上次執行儲存的快照與券商目前的部位及待處理委託
<!-- ML4T_ZH_END c0020-p000 -->


In [12]:
RISK_STATE_PATH = get_output_dir(25, "ib_paper_demo") / "risk_state.json"
risk_config = LiveRiskConfig(
    shadow_mode=True,  # CRITICAL: Virtual orders only!
    max_position_value=50_000.0,
    max_order_value=10_000.0,
    max_orders_per_minute=5,
    max_daily_loss=2_500.0,
    max_data_staleness_seconds=60,
    dedup_window_seconds=0.0,  # Disable for demo
    state_file=str(RISK_STATE_PATH),
)

safe_broker = SafeBroker(broker, risk_config)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
run_async(safe_broker.connect())

print("\n" + "=" * 60)
print("RISK CONFIGURATION (SHADOW MODE)")
print("=" * 60)
print("   Shadow Mode: [OK] ENABLED (no real orders)")
print(f"   Max Position Value: ${risk_config.max_position_value:,.0f}")
print(f"   Max Order Value: ${risk_config.max_order_value:,.0f}")
print(f"   Max Daily Loss: ${risk_config.max_daily_loss:,.0f}")
print(f"   Max Data Staleness: {risk_config.max_data_staleness_seconds}s")
print(f"   Rate Limit: {risk_config.max_orders_per_minute}/minute")
print(f"   Risk State: {display_path(RISK_STATE_PATH)}")
report = safe_broker.reconciliation_report
print(f"   Startup Reconciliation: {'clean' if report and report['clean'] else 'review report'}")

2026-07-23 10:18:23,273 - ml4t.live.safety - INFO - SafeBroker initialized. Shadow mode: True


2026-07-23 10:18:23,276 - ml4t.live.safety - INFO - SafeBroker reconciliation clean



RISK CONFIGURATION (SHADOW MODE)
   Shadow Mode: [OK] ENABLED (no real orders)
   Max Position Value: $50,000
   Max Order Value: $10,000
   Max Daily Loss: $2,500
   Max Data Staleness: 60s
   Rate Limit: 5/minute
   Risk State: <ML4T_OUTPUT_DIR>/ch25_ib_paper_demo/risk_state.json
   Startup Reconciliation: clean


## 5. Real-Time Data Feed

`IBDataFeed` subscribes to live tick data from IB; `BarAggregator` rolls those
ticks into minute bars for the strategy. A live notebook is only meaningful
while the market is open. If the run starts after the close, the
notebook fails loudly rather than substituting historical bars (a backtest
wearing a live disguise).

<!-- ML4T_ZH_BEGIN c0022-p000 -->
## 5. 即時資料來源

`IBDataFeed` 訂閱 IB 的即時逐筆資料；`BarAggregator` 將這些資料彙整為策略使用的分鐘價格長條。即時 Notebook 只有在市場開盤時才有意義。如果執行在收盤後開始，Notebook 會明確失敗，而非以歷史資料代替價格長條資料（使回測偽裝成實盤）。
<!-- ML4T_ZH_END c0022-p000 -->


In [13]:
def _is_rth_now(zone: str = "America/New_York") -> bool:
    """Return True iff wall-clock now is inside US-equity regular trading hours."""
    from zoneinfo import ZoneInfo

    now = datetime.now(ZoneInfo(zone))
    if now.weekday() >= 5:
        return False
    open_t = now.replace(hour=9, minute=30, second=0, microsecond=0)
    close_t = now.replace(hour=16, minute=0, second=0, microsecond=0)
    return open_t <= now <= close_t

The feed, aggregator, and engine are assembled in one helper so the bounded
run below focuses only on lifecycle and cleanup.

<!-- ML4T_ZH_BEGIN c0024-p000 -->
資料來源、彙整器及引擎在一個輔助函式中組裝，讓下方有時間限制的執行只需處理生命週期與清理工作。
<!-- ML4T_ZH_END c0024-p000 -->


In [14]:
def build_live_stack() -> tuple[BarAggregator, LiveEngine]:
    """Build the IB feed, minute aggregator, and shadow execution engine."""
    ib_feed = IBDataFeed(ib=broker.ib, symbols=SYMBOLS, tick_throttle_ms=1000)
    feed = BarAggregator(source_feed=ib_feed, bar_size_minutes=1, assets=SYMBOLS)
    strategy = MomentumStrategy(lookback=5, threshold=0.02)
    engine = LiveEngine(strategy=strategy, broker=safe_broker, feed=feed)
    return feed, engine

A fixed-duration RTH run proves connectivity without leaving an unattended
strategy loop behind.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
在一般交易時段（RTH）執行固定時長，能驗證連線，又不會留下無人看管的策略循環。
<!-- ML4T_ZH_END c0026-p000 -->


In [15]:
async def run_live_demo(duration_seconds: int = 30):
    """Run strategy with live IB data for specified duration."""
    print("\n" + "=" * 60)
    print("LIVE TRADING DEMO (Shadow Mode)")
    print("=" * 60)

    if not _is_rth_now():
        print()
        print("ERROR: Market is closed. This notebook requires an open RTH session.")
        print("US equity RTH: 09:30-16:00 America/New_York, Mon-Fri.")
        print("Re-run during RTH; the notebook will not synthesize live behaviour from history.")
        raise RuntimeError("IB live-feed gate requires an open US-equity RTH session")

    feed, engine = build_live_stack()
    print(f"\nStarting live engine for {duration_seconds} seconds...")
    print("   Watching: " + ", ".join(SYMBOLS))

    engine_task: asyncio.Task | None = None
    try:
        await engine.connect()

        # Run for a fixed duration. `asyncio.wait_for` is unreliable under
        # nest_asyncio in notebook kernels, so we drive the timeout manually.
        engine_task = asyncio.create_task(engine.run())
        await asyncio.sleep(duration_seconds)
        print(f"\nDemo duration ({duration_seconds}s) reached")
    finally:
        if engine_task is not None and not engine_task.done():
            engine_task.cancel()
            try:
                await engine_task
            except asyncio.CancelledError:
                pass
        feed.stop()
        print("\nEngine Statistics:")
        for key, value in engine.stats.items():
            print(f"   {key}: {value}")


# The live workflow runs after the shadow-order routine is defined so one
# outer `finally` block can always release the IB session.

The strategy and the risk configuration are the same whether or not the live feed is active;
only the transport changes. Keeping that boundary is what makes the stack testable, and it means
a change of feed is never a change of trading rule.

<!-- ML4T_ZH_BEGIN c0028-p000 -->
無論即時資料來源是否運作，策略與風險設定都相同；改變的只有傳輸方式。維持這個邊界，整套系統才可測試，也表示更換資料來源不會改變交易規則。
<!-- ML4T_ZH_END c0028-p000 -->


## 6. Order Submission Demo

Let's demonstrate order submission in shadow mode.
Orders are tracked virtually but never sent to IB.

<!-- ML4T_ZH_BEGIN c0029-p000 -->
## 6. 送出委託示範

以下示範在影子模式下送出委託。委託會以虛擬方式追蹤，絕不送往 IB。
<!-- ML4T_ZH_END c0029-p000 -->


In [16]:
async def fetch_ib_snapshot(symbol: str) -> float | None:
    """Fetch one delayed top-of-book snapshot from IB for `symbol` and return mid price."""
    contract = Stock(symbol, "SMART", "USD")
    qualified = await broker.ib.qualifyContractsAsync(contract)
    if not qualified:
        return None
    ticker = broker.ib.reqMktData(qualified[0], "", snapshot=True, regulatorySnapshot=False)
    deadline = datetime.now().timestamp() + 5.0
    while datetime.now().timestamp() < deadline:
        bid = float(ticker.bid) if ticker.bid and ticker.bid > 0 else None
        ask = float(ticker.ask) if ticker.ask and ticker.ask > 0 else None
        last = float(ticker.last) if ticker.last and ticker.last > 0 else None
        close = float(ticker.close) if ticker.close and ticker.close > 0 else None
        if bid and ask:
            return (bid + ask) / 2
        if last:
            return last
        if close:
            return close
        await asyncio.sleep(0.2)
    return None

Every order below is preceded by a real snapshot quote pulled from IB and recorded on
`SafeBroker`, never a research-time price and never an in-memory mock. That ordering is the
operational discipline the chapter is teaching. Delayed data is enough to demonstrate it;
whether the account carries live subscriptions is an account-level question.

The two quantities are sized to fit `max_order_value` at current SPY and QQQ levels, so the
demo shows a virtual fill rather than the cap rejection. Every other control still applies to
each leg.

<!-- ML4T_ZH_BEGIN c0031-p000 -->
下方每筆委託之前，都會先從 IB 擷取真正的快照報價，並記錄於 `SafeBroker`；不使用研究階段的價格，也不使用記憶體中的模擬資料。這個順序就是本章要教授的作業紀律。延遲資料足以示範此流程；帳戶是否具有即時訂閱，則取決於帳戶層級設定。

兩筆委託的數量依目前 SPY 與 QQQ 價位設定，足以符合 `max_order_value`，因此示範會顯示虛擬成交，而非因超過上限遭拒。其他控制仍適用於每筆委託。
<!-- ML4T_ZH_END c0031-p000 -->


In [17]:
async def demonstrate_order_submission():
    """Show order submission in shadow mode using live IB snapshot quotes."""
    print("\n" + "=" * 60)
    print("ORDER SUBMISSION DEMO (Shadow Mode)")
    print("=" * 60)

    print("\nFetching delayed snapshot quotes from IB...")
    snapshot_prices = {}
    for symbol in SYMBOLS:
        price = await fetch_ib_snapshot(symbol)
        if price is None:
            raise RuntimeError(f"No IB snapshot quote received for {symbol} within 5 seconds")
        safe_broker.record_market_snapshot(symbol, price)
        snapshot_prices[symbol] = price
        print(f"   {symbol}: ${price:,.2f}")
    assert set(snapshot_prices) == set(SYMBOLS)

    orders = [("SPY", 10), ("QQQ", 12)]

    for symbol, qty in orders:
        print(f"\nSubmitting virtual BUY order: {qty} shares {symbol}")
        order = await safe_broker.submit_order_async(symbol, qty, side=OrderSide.BUY)
        print(f"   Order ID: {order.order_id}")
        print(f"   Status: {order.status.value}")
        print(f"   Side: {order.side.value}")
        print(f"   Quantity: {order.quantity}")

    vp = safe_broker._virtual_portfolio
    print("\nFinal Virtual Portfolio:")
    print(f"   Cash: ${vp.cash:,.2f}")
    for symbol, pos in vp.positions.items():
        value = pos.quantity * (pos.current_price or pos.entry_price)
        print(f"   {symbol}: {pos.quantity} shares @ ${pos.entry_price:.2f} = ${value:,.2f}")

One outer cleanup boundary covers both the live-feed demonstration and the
virtual-order demonstration.

<!-- ML4T_ZH_BEGIN c0033-p000 -->
同一層外部清理邊界涵蓋即時資料來源示範與虛擬委託示範。
<!-- ML4T_ZH_END c0033-p000 -->


In [18]:
async def run_shadow_workflow() -> None:
    """Run the live-feed and shadow-order gates with guaranteed cleanup."""
    try:
        await run_live_demo(duration_seconds=LIVE_DURATION_SECONDS)
        await demonstrate_order_submission()
    finally:
        await safe_broker.disconnect()


warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
run_async(run_shadow_workflow())

2026-07-23 10:18:23,371 - ml4t.live.engine - INFO - LiveEngine: Connecting...


2026-07-23 10:18:23,373 - ml4t.live.safety - INFO - SafeBroker reconciliation clean


2026-07-23 10:18:23,374 - ml4t.live.feeds.ib_feed - INFO - IBDataFeed: Starting feed for 3 symbols



LIVE TRADING DEMO (Shadow Mode)

Starting live engine for 75 seconds...
   Watching: SPY, QQQ, IWM


2026-07-23 10:18:23,753 - ml4t.live.feeds.ib_feed - INFO - IBDataFeed: Subscribed to 3 symbols


2026-07-23 10:18:23,760 - ml4t.live.engine - INFO - LiveEngine: Connected and ready


... (17 outputs left out)

2026-07-23 10:19:39,395 - ml4t.live.safety - INFO - SHADOW: buy 10 SPY @ $740.44 (value: $7,404)


2026-07-23 10:19:39,396 - ml4t.live.safety - INFO - Shadow: Opened QQQ LONG 12


2026-07-23 10:19:39,396 - ml4t.live.safety - INFO - SHADOW: buy 12 QQQ @ $695.74 (value: $8,349)


   IWM: $292.17

Submitting virtual BUY order: 10 shares SPY
   Order ID: SHADOW-1784816379394
   Status: filled
   Side: buy
   Quantity: 10

Submitting virtual BUY order: 12 shares QQQ
   Order ID: SHADOW-1784816379396
   Status: filled
   Side: buy
   Quantity: 12

Final Virtual Portfolio:
   Cash: $84,246.78
   SPY: 10 shares @ $740.44 = $7,404.40
   QQQ: 12 shares @ $695.74 = $8,348.82


The shadow portfolio is printed after each submission, so the path from intent to inventory is
visible without capital at risk. That path is where production bugs surface first, because the
strategy, the broker adapter and the risk guard all interact on it at once.

<!-- ML4T_ZH_BEGIN c0035-p000 -->
每次送單後都會印出影子投資組合，因此可在資金不承受風險的情況下，看見從交易意圖到持倉的流程。生產環境中的錯誤往往最先出現在此流程，因為策略、券商介接器及風險防護會在此同時互動。
<!-- ML4T_ZH_END c0035-p000 -->


## 7. Clean Shutdown

Always disconnect cleanly to avoid connection issues on the next run. A disciplined shutdown sequence is
part of live reliability because stale sessions and dangling subscriptions are operational bugs too.

<!-- ML4T_ZH_BEGIN c0036-p000 -->
## 7. 正常關閉

務必正常中斷連線，以免下次執行發生連線問題。有紀律的關閉順序是實盤可靠性的一部分，因為過時的工作階段與未解除的訂閱同樣是作業錯誤。
<!-- ML4T_ZH_END c0036-p000 -->


In [19]:
print("\n[OK] Shadow workflow completed and disconnected from IB")


[OK] Shadow workflow completed and disconnected from IB


## Summary

This notebook demonstrated the complete IB integration workflow:

1. **Connection**: Connect to TWS/Gateway paper trading
2. **Account Info**: Query NLV, cash, positions
3. **Historical Data**: Request bars for strategy warm-up
4. **Real-Time Feed**: Subscribe to tick data, aggregate to bars
5. **Safe Trading**: Use SafeBroker in shadow mode
6. **Order Submission**: Virtual orders tracked in VirtualPortfolio

### Key Takeaways

- **Same Strategy class** works in backtest and live
- **Shadow mode first** - always test before real trading
- **SafeBroker** provides 8 layers of protection
- **IBBroker** handles all IB-specific details

### Next Steps

1. Run in shadow mode for 1-2 weeks
2. Verify signals match backtest expectations
3. Enable paper trading (`shadow_mode=False`)
4. Monitor for 2-4 weeks on paper
5. Gradually transition to live with small positions

<!-- ML4T_ZH_BEGIN c0038-p000 -->
## 摘要

本 Notebook 示範完整的 IB 整合流程：

1. **連線**：連接 TWS／Gateway 模擬交易
2. **帳戶資訊**：查詢淨清算價值（NLV）、現金與部位
3. **歷史資料**：請求價格長條資料供策略暖身
4. **即時資料來源**：訂閱逐筆資料並彙整成價格長條資料
5. **安全交易**：以影子模式使用 SafeBroker
6. **送出委託**：在 VirtualPortfolio 中追蹤虛擬委託

### 重點摘要

- **同一個 Strategy 類別**可用於回測與實盤
- **先使用影子模式**：進行真實交易前務必測試
- **SafeBroker** 提供八層防護
- **IBBroker** 處理所有 IB 專屬細節

### 後續步驟

1. 在影子模式下執行 1 至 2 週
2. 驗證訊號是否符合回測預期
3. 啟用模擬交易（`shadow_mode=False`）
4. 在模擬交易環境監控 2 至 4 週
5. 逐步以小額部位轉入實盤
<!-- ML4T_ZH_END c0038-p000 -->


In [20]:
print("\n" + "=" * 60)
print("DEMO COMPLETE")
print("=" * 60)
print("Paper account readiness: verified")
print(f"Symbols Monitored: {', '.join(SYMBOLS)}")
print("Shadow Mode: ENABLED [OK]")
print("\nThe same MomentumStrategy that runs in backtest")
print("completed the IB shadow workflow without sending an order to the venue.")


DEMO COMPLETE
Paper account readiness: verified
Symbols Monitored: SPY, QQQ, IWM
Shadow Mode: ENABLED [OK]

The same MomentumStrategy that runs in backtest
completed the IB shadow workflow without sending an order to the venue.


## Key Takeaways

IB carries more connectivity and market-structure complexity than Alpaca, and the execution
architecture above absorbs all of it without the strategy knowing. Once connectivity, warm-up
and the shadow-mode controls are standard, moving a strategy between brokers is a configuration
change rather than a rewrite.

**Next**: Compare the simpler REST-style path in `04_alpaca_paper_trading_demo`, then move to
`08_pipeline_verification` to check that research outputs and live inputs still match.

<!-- ML4T_ZH_BEGIN c0040-p000 -->
## 重點摘要

IB 的連線與市場結構比 Alpaca 複雜，但上方執行架構會吸收這些複雜性，策略本身不必知悉。一旦連線、暖身及影子模式控制都標準化，在券商之間移動策略就成為設定變更，而非重寫。

**下一步**：在 `04_alpaca_paper_trading_demo` 比較較簡單的 REST 風格路徑，接著前往 `08_pipeline_verification` 檢查研究輸出與即時輸入是否仍一致。
<!-- ML4T_ZH_END c0040-p000 -->
